### The working of the tool is described below.
* Takes a LeetCode problem link as input
* Extracts the problem description using web scraping (BeautifulSoup)
* Cleans and structures the raw extracted content
* Sends the structured data through an API for refinement/formatting
* Uses the processed output to generate a complete solution package
* Produces:

  * Clear problem explanation
  * Intuition behind the approach
  * Complexity analysis
  * C++ implementation
  * Step-by-step dry run


In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import requests
from bs4 import BeautifulSoup
import re

In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AIz') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'

In [ ]:
"""Code to run the model locally using ollama, you can ignore this if you're using gemini"""

# MODEL = 'llama3.2'
# OLLAMA_BASE_URL = "http://localhost:11434/v1"
# gemini = OpenAI(base_url=OLLAMA_BASE_URL, api_key="anything")


"""Code to run the model using gemini, you can ignore this if you're using ollama"""

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

In [ ]:
#The functions below extract the problem description using web scraping (BeautifulSoup)

def structured_extract(html):
    soup = BeautifulSoup(html, "html.parser")

    # Fix superscripts
    for sup in soup.find_all("sup"):
        sup.string = "^" + sup.get_text()

    sections = []

    for tag in soup.find_all(['p', 'pre', 'li']):
        text = tag.get_text(" ", strip=True)
        if text:
            sections.append(text)

    return "\n".join(sections)


def get_problem_description(slug):
    url = "https://leetcode.com/graphql"

    query = {
        "operationName": "questionData",
        "variables": {"titleSlug": slug},
        "query": """
        query questionData($titleSlug: String!) {
          question(titleSlug: $titleSlug) {
            title
            content
            difficulty
          }
        }
        """
    }

    headers = {
        "Content-Type": "application/json",
        "Referer": f"https://leetcode.com/problems/{slug}/",
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.post(url, json=query, headers=headers)
    data = response.json()

    html_content = data["data"]["question"]["content"]

    # 🔥 Apply cleaning here
    clean_description = structured_extract(html_content)
    
    #print(data["data"]["question"]["title"])
    
    return str(clean_description)

In [ ]:
# Example of how to use the above function
problem = "two-sum"
description = get_problem_description(problem)
description

In [ ]:
#Leetcode problem description formatting assistant's system and user prompt

desc_system_prompt = """
You are a formatter and just format content without any ommisions or additions.

Reformat the following DSA problem into clean structured text.

Rules:
- Keep content EXACTLY same (do not modify meaning)
- Add proper line breaks
- Separate sections:
  - Description
  - Examples
  - Constraints
  - Follow-up if given
- Fix spacing issues like "target ," → "target,"

"""


def get_desc_user_prompt(problem):
    user_prompt = f"""
Below is the content of the description of the dsa problem from leetcode.
Format the description so as it looks clean and structured. The description is as follows:
"""
    user_prompt+=get_problem_description(problem)
    return user_prompt

In [ ]:
#API call to get the formatted problem description from the model

def problem_description(problem):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": desc_system_prompt},
            {"role": "user", "content": get_desc_user_prompt(problem)}
        ]
    )
    result = response.choices[0].message.content
    return result
    

In [ ]:
#Example of how to use the above API call to get the formatted problem description

print(problem_description("two-sum"))

In [ ]:
#Leetcode problem solving assistant's system and user prompt

dsa_system_prompt = """
You are an expert Data Structures and Algorithms (DSA) tutor and competitive programmer.

Your goal is to help solve coding problems from LeetCode platform with maximum clarity and correctness.

Follow this exact structure in every response:

1. Problem Understanding
- Restate the problem in simple words

2. Intuition
- Explain the thought process step-by-step
- Start from brute force idea, then optimize
- Explain WHY the optimal approach works

3. Complexity Analysis
- Time Complexity (Big-O)
- Space Complexity (Big-O)
- Brief justification

4. Code (C++ preferred)
- Clean, readable, and optimal code
- Use meaningful variable names
- No unnecessary comments

5. Dry Run
- Walk through the sample input step-by-step
- Show how variables change

Rules:
- Always prefer optimal solutions over brute force
- If multiple approaches exist, briefly compare them
- Be concise but clear (avoid unnecessary verbosity)
- Do NOT hallucinate constraints or inputs
- If problem is ambiguous, ask for clarification

Tone:
- Clear, structured, and educational
- Think like a top competitive programming mentor
"""


def get_dsa_user_prompt(problem):
    user_prompt = f"""
Below is the content of the description of the dsa problem from leetcode.
Solve the following DSA problem. {problem_description(problem)}

Requirements:
- Follow the structured format strictly
- Focus on optimal solution
- Use C++ for implementation
- Include dry run on given example
- Keep explanation clear but not too verbose
"""
    return user_prompt

In [ ]:
#API call to get the solution to the problem in a structured format as per the system prompt defined above

def solve_problem(problem):
    print(f"Solving the dsa probelm {problem} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": dsa_system_prompt},
            {"role": "user", "content": get_dsa_user_prompt(problem)}
        ]
    )
    result = response.choices[0].message.content
    
    print(f"Solved the problem")
    return result


In [ ]:
#Example of for final API call to get the solution to the problem in a structured format as per the system prompt defined above

result = solve_problem("two-sum")

Markdown(result)